# 03 - Image Modelling Baseline

This notebook trains and evaluates a minimal image-only baseline on the clean shared split. It compares a majority-class baseline with a pretrained ResNet18 whose classification head is trained for the 27 Rakuten product classes.

## 1. Evaluation Contract

Training uses only rows marked `train` in `image_split_clean.csv`. Scores are computed only on rows marked `validation`. The official challenge test set is excluded because it has no labels. Weighted F1 is the primary score and macro F1 is reported to expose performance on smaller classes.

## 2. Imports and Configuration

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

from src.data_utils import get_data_paths, get_image_path

RANDOM_STATE    = 42
BATCH_SIZE      = 32
EPOCHS          = 1
LEARNING_RATE   = 1e-3
NUM_WORKERS     = 0

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")


Device: mps


## 3. Load the Clean Manifest and Encode the Labels

The Rakuten class codes are mapped deterministically to indices from 0 to 26 for `CrossEntropyLoss`. Image paths are reconstructed locally and are not stored in the portable manifest.

In [ ]:
data_paths      = get_data_paths()
processed_dir   = Path(data_paths["x_train"]).parent.parent / "processed"
manifest_path   = processed_dir / "image_split_clean.csv"

if not manifest_path.exists():
    raise FileNotFoundError(
        "Clean image manifest not found. Run notebooks/02_preprocessing_image.ipynb first."
    )

image_manifest      = pd.read_csv(manifest_path)
required_columns    = {"row_id", "imageid", "productid", "prdtypecode", "split"}
assert required_columns.issubset(image_manifest.columns)
assert image_manifest["row_id"].is_unique
assert set(image_manifest["split"]) == {"train", "validation"}

class_codes = sorted(image_manifest["prdtypecode"].astype(int).unique().tolist())
assert len(class_codes) == 27
class_to_index = {code: index for index, code in enumerate(class_codes)}
index_to_class = {index: code for code, index in class_to_index.items()}

image_manifest["target"    ] = image_manifest["prdtypecode"].astype(int).map(class_to_index)
image_manifest["image_path"] = [
    Path(get_image_path(int(row.imageid), int(row.productid), split="image_train"))
    for row in image_manifest.itertuples(index=False)
]
assert image_manifest["target"    ].notna().all()
assert image_manifest["image_path"].map(Path.exists).all()

train_manifest      = image_manifest.loc[image_manifest["split"].eq("train")].reset_index(drop=True)
validation_manifest = image_manifest.loc[image_manifest["split"].eq("validation")].reset_index(drop=True)

display(pd.DataFrame({
    "rows": [len(train_manifest), len(validation_manifest)],
    "classes": [train_manifest["target"].nunique(), validation_manifest["target"].nunique()],
}, index=["train", "validation"]))


,rows,classes
train,62312,27
validation,16433,27


## 4. Majority-Class Baseline

This baseline predicts the most frequent training class for every validation product. It provides the minimum reference score that the image model must improve upon.

In [ ]:
majority_target         = int(train_manifest["target"].mode().iloc[0])
validation_targets      = validation_manifest["target"].astype(int).to_numpy()
majority_predictions    = np.full(len(validation_targets), majority_target)

majority_metrics = {
    "accuracy": accuracy_score(validation_targets, majority_predictions),
    "weighted_f1": f1_score(
        validation_targets, majority_predictions, average="weighted", zero_division=0
    ),
    "macro_f1": f1_score(
        validation_targets, majority_predictions, average="macro", zero_division=0
    ),
}

print(f"Majority class: {index_to_class[majority_target]}")
display(pd.DataFrame([majority_metrics], index=["majority_class"]).style.format("{:.4f}"))


Majority class: 2583


,accuracy,weighted_f1,macro_f1
majority_class,0.1122,0.0226,0.0075


## 5. Image Dataset and DataLoaders

The default ResNet18 preprocessing resizes, crops and normalizes images using the ImageNet configuration associated with the pretrained weights. The same deterministic transform is used for this first baseline.

In [ ]:
weights         = ResNet18_Weights.DEFAULT
image_transform = weights.transforms()


class RakutenImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform):
        self.frame      = frame.reset_index(drop=True)
        self.transform  = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["image_path"]) as image:
            image_tensor = self.transform(image.convert("RGB"))
        return image_tensor, int(row["target"]), int(row["row_id"])


train_dataset       = RakutenImageDataset(train_manifest, image_transform)
validation_dataset  = RakutenImageDataset(validation_manifest, image_transform)

loader_generator    = torch.Generator().manual_seed(RANDOM_STATE)
train_loader        = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    generator   = loader_generator,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
)

print(f"Training batches: {len(train_loader):,}")
print(f"Validation batches: {len(validation_loader):,}")


Training batches: 1,948
Validation batches: 514


## 6. Configure the ResNet18 Baseline

The pretrained feature extractor is frozen and only the final classification layer is trained. This keeps the first modelling iteration fast and provides a clear baseline before fine-tuning or augmentation.


In [8]:
model = resnet18(weights=weights)
for parameter in model.parameters():
    parameter.requires_grad = False

model.fc  = nn.Linear(model.fc.in_features, len(class_codes))
model     = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=LEARNING_RATE)

trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print(f"Trainable parameters: {trainable_parameters:,}")


Trainable parameters: 13,851


## 7. Train the Classification Head

In [6]:
training_history = []

for epoch in range(EPOCHS):
    model.eval()
    model.fc.train()
    running_loss    = 0.0
    processed_rows  = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
    for images, targets, _ in progress:
        images  = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits  = model(images)
        loss    = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        batch_rows       = targets.size(0)
        running_loss    += loss.item() * batch_rows
        processed_rows  += batch_rows
        progress.set_postfix(loss=f"{running_loss / processed_rows:.4f}")

    epoch_loss = running_loss / processed_rows
    training_history.append({"epoch": epoch + 1, "train_loss": epoch_loss})

display(pd.DataFrame(training_history).set_index("epoch").style.format("{:.4f}"))


Epoch 1/1:   0%|          | 0/1948 [00:00<?, ?it/s]

,train_loss
epoch,
1,1.8135


## 8. Evaluate and Save the Baseline

The checkpoint, label mapping and validation predictions are saved for reproducibility and later multimodal fusion. The prediction file retains `row_id` and all 27 class probabilities.

In [ ]:
model.eval()
all_row_ids             = []
all_targets             = []
all_probabilities       = []
validation_loss_total   = 0.0

with torch.no_grad():
    for images, targets, row_ids in tqdm(validation_loader, desc="Validation"):
        images  = images.to(device)
        targets = targets.to(device)
        logits  = model(images)
        validation_loss_total += criterion(logits, targets).item() * targets.size(0)

        all_row_ids.extend(row_ids.tolist())
        all_targets.extend(targets.cpu().tolist())
        all_probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())

probabilities       = np.concatenate(all_probabilities, axis=0)
true_targets        = np.asarray(all_targets)
predicted_targets   = probabilities.argmax(axis=1)

resnet_metrics = {
    "accuracy   ": accuracy_score(true_targets, predicted_targets),
    "weighted_f1": f1_score(
        true_targets, predicted_targets, average="weighted", zero_division=0
    ),
    "macro_f1   ": f1_score(
        true_targets, predicted_targets, average="macro", zero_division=0
    ),
}

score_table = pd.DataFrame( [majority_metrics, resnet_metrics], index=["majority_class", "resnet18_frozen"] )
display(score_table.style.format("{:.4f}"))
print(f"Validation loss: {validation_loss_total / len(validation_dataset):.4f}")

project_root    = Path(data_paths["x_train"]).parents[2]
model_dir       = project_root / "models"

model_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = model_dir / "resnet18_image_baseline.pt"
mapping_path    = model_dir / "resnet18_class_mapping.json"
prediction_path = processed_dir / "image_validation_predictions.csv"

cpu_state_dict  = {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()}
torch.save({
    "model_name         ": "resnet18",
    "model_state_dict   ": cpu_state_dict,
    "class_to_index     ": class_to_index,
    "validation_metrics ": resnet_metrics,
}, checkpoint_path)

mapping_path.write_text(
    json.dumps({str(index): code for index, code in index_to_class.items()}, indent=2),
    encoding="utf-8",
)

prediction_frame = pd.DataFrame({
    "row_id                 ": all_row_ids,
    "true_prdtypecode       ": [index_to_class[index] for index in true_targets],
    "predicted_prdtypecode  ": [index_to_class[index] for index in predicted_targets],
})
for index, class_code in index_to_class.items():
    prediction_frame[f"prob_{class_code}"] = probabilities[:, index]
prediction_frame.to_csv(prediction_path, index=False)

assert prediction_frame["row_id"].is_unique
assert len(prediction_frame) == len(validation_manifest)

print(f"Saved checkpoint            : {checkpoint_path}")
print(f"Saved class mapping         : {mapping_path}")
print(f"Saved validation predictions: {prediction_path}")


Validation:   0%|          | 0/514 [00:00<?, ?it/s]

,accuracy,weighted_f1,macro_f1
majority_class,0.1122,0.0226,0.0075
resnet18_frozen,0.5012,0.4813,0.4241


Validation loss: 1.6670
Saved checkpoint: /Users/Ben/Development/VS-Python/Rakuten2/liora-capstone/models/resnet18_image_baseline.pt
Saved class mapping: /Users/Ben/Development/VS-Python/Rakuten2/liora-capstone/models/resnet18_class_mapping.json
Saved validation predictions: /Users/Ben/Development/VS-Python/Rakuten2/liora-capstone/data/processed/image_validation_predictions.csv
